<a href="https://colab.research.google.com/github/pari12-1/ABTalks/blob/main/Day_7_sentiment_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import nltk  #Natural Language Toolkit  [ provides tools and datasets for working with text ]

from nltk.corpus import movie_reviews

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [2]:
nltk.download("movie_reviews")

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


True

In [6]:
documents = []

for fileid in movie_reviews.fileids():
    words = movie_reviews.words(fileid)
    text = " ".join(words)

    if movie_reviews.categories(fileid)[0] == "pos":
        label = "positive"
    else:
        label = "negative"

    documents.append((text, label))

df = pd.DataFrame(documents, columns=["review", "sentiment"])

df.head()

,review,sentiment
0,"plot : two teen couples go to a church party ,...",negative
1,the happy bastard ' s quick movie review damn ...,negative
2,it is movies like these that make a jaded movi...,negative
3,""" quest for camelot "" is warner bros . ' first...",negative
4,synopsis : a mentally unstable man undergoing ...,negative


In [7]:
print("Number of reviews:", len(df))
print(df["sentiment"].value_counts())

Number of reviews: 2000
sentiment
negative    1000
positive    1000
Name: count, dtype: int64


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

In [10]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=10000
    )),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [12]:
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, stop_words='english')),
                ('classifier', LogisticRegression(max_iter=1000))])

In [14]:
predictions = model.predict(X_test)

In [15]:
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.825


In [16]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

    negative       0.84      0.81      0.82       200
    positive       0.81      0.84      0.83       200

    accuracy                           0.82       400
   macro avg       0.83      0.82      0.82       400
weighted avg       0.83      0.82      0.82       400



In [17]:
test_sentences = [
    "I absolutely loved this movie. It was fantastic!",
    "This was one of the worst movies I have ever watched.",
    "The movie was okay, but nothing special.",
    "The acting was excellent and the story was beautiful.",
    "I expected a lot more from this film and was disappointed."
]

predictions = model.predict(test_sentences)

for sentence, prediction in zip(test_sentences, predictions):
    print(f"Review: {sentence}")
    print(f"Prediction: {prediction}")
    print()

Review: I absolutely loved this movie. It was fantastic!
Prediction: positive

Review: This was one of the worst movies I have ever watched.
Prediction: negative

Review: The movie was okay, but nothing special.
Prediction: negative

Review: The acting was excellent and the story was beautiful.
Prediction: positive

Review: I expected a lot more from this film and was disappointed.
Prediction: positive



In [18]:
test_sentence_1 = "I absolutely loved this movie. It was fantastic!"
predictions = model.predict([test_sentence_1])
print(f"Review: {test_sentence_1}")
print(f"Prediction: {predictions}")

Review: I absolutely loved this movie. It was fantastic!
Prediction: ['positive']


In [19]:
test_sentence_2 = "This was one of the worst movies I have ever watched."
predictions = model.predict([test_sentence_2])
print(f"Review: {test_sentence_2}")
print(f"Prediction: {predictions}")

Review: This was one of the worst movies I have ever watched.
Prediction: ['negative']


In [20]:
test_sentence_3 = "The movie was okay, but nothing special."
predictions = model.predict([test_sentence_3])
print(f"Review: {test_sentence_3}")
print(f"Prediction: {predictions}")
#

Review: The movie was okay, but nothing special.
Prediction: ['negative']


In [21]:
test_sentence_4 = "The acting was excellent and the story was beautiful."
predictions = model.predict([test_sentence_4])
print(f"Review: {test_sentence_4}")
print(f"Prediction: {predictions}")
#

Review: The acting was excellent and the story was beautiful.
Prediction: ['positive']


In [24]:
test_sentence_5 = "I expected a lot more from this film and was disappointed."
predictions = model.predict([test_sentence_5])
print(f"Review: {test_sentence_5}")
print(f"Prediction: {predictions}")
# here is the prediction should be "negative".


Review: I expected a lot more from this film and was disappointed.
Prediction: ['positive']


In [25]:
test_sentence_6 = "Wow, what an amazing movie... if you enjoy falling asleep."
predictions = model.predict([test_sentence_6])
print(f"Review: {test_sentence_6}")
print(f"Prediction: {predictions}")
# here is the prediction should be "negative

Review: Wow, what an amazing movie... if you enjoy falling asleep.
Prediction: ['positive']


# where the model fails ?
# reflection

the model doesn't know what we are feeding it . it only learns the statistical relationships between words and sentiments. it learns patterns from these only.

if we observe sentence 5 and 6 ... we can see , how it makes mistake sometimes ... eventhough it is written " disappointed " in sentence 5 . it still predicts it as positive ...


in sentence 6 , humans understands that the movie was boring , but the model sees " amazing " which strongly correlates to "positive" . therefore it predicts positive. .. the model don't understand sarcasm.


also some mixed opinion are hard to predict like :
1. "I don't think this movie was good." ... [ good , which can may predict positive " ]
2. "The acting was fantastic, but the story was terrible."
3. "The first half was boring, but the second half was incredible."
   
   model can get confuse , while predicting this .

• These limitations occur because TF-IDF mainly represents words based on their statistical importance and does not fully understand context, sarcasm, or the meaning of a sentence.
